# 09 — Optional English-trained anchor for forgetting

**Purpose:** Create one shared English-specialized model from which BOTH LoRA and full Urdu adaptation begin.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


A direct before/after English score from the original Qwen model is only a language-retention check, not necessarily meaningful *catastrophic forgetting*. For a stronger claim, establish English ability first using the SIB-200 English training split. Then save a **shared English-trained checkpoint**, and start both Urdu training methods from that checkpoint. The resulting comparison isolates what Urdu adaptation changes.

In [ ]:
import torch
RUN_ANCHOR=False  # English training consumes additional GPU time
if RUN_ANCHOR:
    assert torch.cuda.is_available()
    r=train_run('lora','sib200',size=10,seed=42,lang='en',rank=8,
                run_name='sib200_english_anchor_lora_r8')
    print('English adapter ready:',r['saved_path'])

## Merge the English LoRA adapter for a common starting point

This modifies the base weights once, creating a new English-adapted checkpoint. Then both Urdu methods begin from the same saved merged model. **Do not mix the English adapter and Urdu adapter in a way that gives only one method a hidden advantage.**

In [ ]:
MERGE_ANCHOR=False
if MERGE_ANCHOR:
    from peft import PeftModel
    r=read_json(ROOT/'outputs/checkpoints/sib200_english_anchor_lora_r8/complete.json')
    m,t=load_model(r['base_model'],adapter_path=r['saved_path'])
    merged=m.merge_and_unload()
    target=ROOT/'outputs/english_anchor_merged'
    target.mkdir(exist_ok=True)
    merged.save_pretrained(str(target),safe_serialization=True)
    t.save_pretrained(str(target))
    print('Saved common anchor:',target)
    print('Set MODEL_OVERRIDE=str(ROOT/"outputs/english_anchor_merged") in notebooks 03 and 04.')

### Cross-lingual evaluation

Use notebook 05 with `LANG='en'` and `SPLIT='test'` to compare the English anchor **before** Urdu adaptation and each Urdu-trained model **after** adaptation. Your final report should distinguish English retention from broader catastrophic forgetting.